# 08 -- Deep Hedging

Neural network-based hedging strategies that minimise risk-adjusted hedging error for European call options.

**Reference:** Buehler, Gonon, Teichmann & Wood (2019), *Deep hedging*, Quantitative Finance 19(8).

---

## Overview

Traditional hedging relies on the Black-Scholes delta, which assumes continuous rebalancing and zero transaction costs. In practice, we rebalance discretely and incur costs. Deep hedging trains a neural network to learn hedging policies that directly minimise a risk measure (e.g. CVaR) of the hedging P&L distribution, accounting for these frictions.

This notebook walks through:
1. Environment setup and GBM path simulation
2. Black-Scholes delta hedging benchmark
3. Training the deep hedger
4. Comparison: BS vs Deep hedging
5. Transaction cost sensitivity analysis

In [ ]:
import sys
from pathlib import Path

# Add project src to path
project_src = str(Path.cwd().parent / "src")
if project_src not in sys.path:
    sys.path.insert(0, project_src)

import matplotlib.pyplot as plt
import numpy as np
import yaml
from diagnostics import (
    plot_hedge_ratio_comparison,
    plot_loss_history,
    plot_pnl_distribution,
    plot_transaction_cost_impact,
)
from environment import HedgingEnvironment
from model import DeepHedgingModel
from network import NeuralNetwork
from trainer import DeepHedgingTrainer

plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 11

# Load config
with open(Path.cwd().parent / "configs" / "default.yaml") as f:
    config = yaml.safe_load(f)
print("Config loaded.")

## 1. Environment Setup and GBM Path Simulation

We simulate stock price paths under Geometric Brownian Motion (risk-neutral measure) for hedging a 1-month ATM European call option with daily rebalancing.

In [ ]:
# Create environment
env = HedgingEnvironment(
    s0=config["option"]["s0"],
    K=config["option"]["K"],
    r=config["option"]["r"],
    sigma=config["option"]["sigma"],
    T=config["option"]["T"],
    n_steps=config["option"]["n_steps"],
    n_paths=config["simulation"]["n_paths_train"],
    cost_rate=config["transaction_cost"]["rate"],
    seed=config["random_seed"],
)

paths = env.simulate_paths()
print(f"Simulated {paths.shape[0]} paths with {paths.shape[1]-1} steps")
print(f"Initial price: {env.s0}, Strike: {env.K}")
print(f"BS option price at t=0: {env.bs_price(env.s0, 0.0):.4f}")

# Fan chart of price paths
fig, ax = plt.subplots(figsize=(10, 5))
time_grid = np.linspace(0, env.T, env.n_steps + 1)

# Plot quantile bands
for q_low, q_high, alpha in [(5, 95, 0.15), (10, 90, 0.2), (25, 75, 0.3)]:
    lo = np.percentile(paths, q_low, axis=0)
    hi = np.percentile(paths, q_high, axis=0)
    ax.fill_between(time_grid, lo, hi, alpha=alpha, color="steelblue")

ax.plot(time_grid, np.median(paths, axis=0), color="steelblue", linewidth=2, label="Median")
ax.axhline(env.K, color="red", linestyle="--", linewidth=1, label=f"Strike K={env.K}")
ax.set_xlabel("Time (years)")
ax.set_ylabel("Stock Price")
ax.set_title("GBM Price Paths -- Fan Chart")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Black-Scholes Delta Hedging Benchmark

We compute the P&L distribution from delta hedging using the Black-Scholes delta. This is the standard benchmark: it is optimal under continuous trading with zero costs.

In [ ]:
# Compute BS delta positions at each step
n_paths = paths.shape[0]
n_steps = paths.shape[1] - 1

bs_positions = np.zeros((n_paths, n_steps))
for t in range(n_steps):
    time_t = t * env.dt
    bs_positions[:, t] = env.bs_delta(paths[:, t], time_t)

bs_pnl = env.compute_pnl(paths, bs_positions)

# No-hedge P&L for comparison
premium = float(env.bs_price(env.s0, 0.0))
no_hedge_pnl = premium - env.compute_payoff(paths[:, -1])

print("--- No Hedge ---")
print(f"  Mean P&L: {np.mean(no_hedge_pnl):.4f}")
print(f"  Std P&L:  {np.std(no_hedge_pnl):.4f}")
print(f"  Min P&L:  {np.min(no_hedge_pnl):.4f}")

print("\n--- BS Delta Hedge ---")
print(f"  Mean P&L: {np.mean(bs_pnl):.4f}")
print(f"  Std P&L:  {np.std(bs_pnl):.4f}")
print(f"  Min P&L:  {np.min(bs_pnl):.4f}")

# Histogram
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(no_hedge_pnl, bins=50, alpha=0.5, label="No Hedge", density=True, color="grey")
ax.hist(bs_pnl, bins=50, alpha=0.5, label="BS Delta Hedge", density=True, color="steelblue")
ax.axvline(0, color="black", linestyle="--", linewidth=0.8)
ax.set_xlabel("Hedging P&L")
ax.set_ylabel("Density")
ax.set_title("P&L Distribution: No Hedge vs BS Delta Hedge")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Train the Deep Hedger

We train a feedforward neural network to learn hedging positions that minimise a risk measure (CVaR or variance) of the P&L distribution. The network takes features `[S_t/S_0, t/T, current_position]` and outputs a delta in `[0, 1]`.

Optimisation uses Natural Evolution Strategies (NES) -- a derivative-free method suitable for our pure-numpy implementation.

In [ ]:
# Build network and trainer
network = NeuralNetwork(
    layer_sizes=config["network"]["layer_sizes"],
    seed=config["random_seed"],
)
print(f"Network parameters: {network.num_parameters()}")

trainer_config = {
    "population_size": config["training"]["population_size"],
    "lr": config["training"]["lr"],
    "risk_measure": config["training"]["risk_measure"],
    "cvar_alpha": config["training"]["cvar_alpha"],
}
trainer = DeepHedgingTrainer(env=env, network=network, config=trainer_config)

# Train (this may take 1-2 minutes)
print("Training... (this may take a minute)")
loss_history = trainer.train(
    n_epochs=config["training"]["n_epochs"],
    lr=config["training"]["lr"],
    measure=config["training"]["risk_measure"],
)
print(f"Training complete. Final loss: {loss_history[-1]:.6f}")

# Loss curve
fig, ax = plot_loss_history(loss_history)
plt.show()

## 4. Comparison: BS vs Deep Hedging

We compare the P&L distributions and hedge ratios of both strategies on a held-out test set.

In [ ]:
# Generate test paths with a different seed
env_test = HedgingEnvironment(
    s0=config["option"]["s0"],
    K=config["option"]["K"],
    r=config["option"]["r"],
    sigma=config["option"]["sigma"],
    T=config["option"]["T"],
    n_steps=config["option"]["n_steps"],
    n_paths=config["simulation"]["n_paths_test"],
    cost_rate=config["transaction_cost"]["rate"],
    seed=config["random_seed"] + 100,
)
test_paths = env_test.simulate_paths()

# BS hedge on test set
n_test = test_paths.shape[0]
n_steps_test = test_paths.shape[1] - 1
bs_test_pos = np.zeros((n_test, n_steps_test))
for t in range(n_steps_test):
    time_t = t * env_test.dt
    bs_test_pos[:, t] = env_test.bs_delta(test_paths[:, t], time_t)
bs_test_pnl = env_test.compute_pnl(test_paths, bs_test_pos)

# Deep hedge on test set
# Point trainer to test env for position computation
trainer.env = env_test
deep_test_pos = trainer.compute_hedge_positions(test_paths)
deep_test_pnl = env_test.compute_pnl(test_paths, deep_test_pos)

print("--- Test Set Results ---")
print(f"BS Delta:   mean={np.mean(bs_test_pnl):.4f}, std={np.std(bs_test_pnl):.4f}")
print(f"Deep Hedge: mean={np.mean(deep_test_pnl):.4f}, std={np.std(deep_test_pnl):.4f}")

# P&L distribution comparison
fig, ax = plot_pnl_distribution(bs_test_pnl, deep_test_pnl)
plt.show()

In [ ]:
# Hedge ratio comparison at t=0
S_range = np.linspace(80, 120, 100)
bs_deltas = env_test.bs_delta(S_range, 0.0)

# Compute deep deltas
deep_deltas = np.zeros(len(S_range))
for i, S in enumerate(S_range):
    features = np.array([[S / env_test.s0, 0.0, 0.5]])  # t=0, position=0.5 (neutral)
    deep_deltas[i] = network.forward(features).ravel()[0]

fig, ax = plot_hedge_ratio_comparison(S_range, bs_deltas, deep_deltas, t=0.0)
plt.show()

## 5. Transaction Cost Sensitivity Analysis

Deep hedging should show larger relative improvement over BS delta hedging as transaction costs increase, since the network can learn to trade less aggressively.

In [ ]:
cost_rates = [0.0, 0.0005, 0.001, 0.002, 0.005, 0.01]
bs_cvars = []
deep_cvars = []

for cr in cost_rates:
    env_cr = HedgingEnvironment(
        s0=config["option"]["s0"],
        K=config["option"]["K"],
        r=config["option"]["r"],
        sigma=config["option"]["sigma"],
        T=config["option"]["T"],
        n_steps=config["option"]["n_steps"],
        n_paths=1000,
        cost_rate=cr,
        seed=config["random_seed"] + 200,
    )
    cr_paths = env_cr.simulate_paths()
    n_p = cr_paths.shape[0]
    n_s = cr_paths.shape[1] - 1

    # BS hedge
    bs_pos = np.zeros((n_p, n_s))
    for t in range(n_s):
        time_t = t * env_cr.dt
        bs_pos[:, t] = env_cr.bs_delta(cr_paths[:, t], time_t)
    bs_pnl_cr = env_cr.compute_pnl(cr_paths, bs_pos)

    # Deep hedge
    trainer.env = env_cr
    deep_pos = trainer.compute_hedge_positions(cr_paths)
    deep_pnl_cr = env_cr.compute_pnl(cr_paths, deep_pos)

    # CVaR computation
    for pnl, cvar_list in [(bs_pnl_cr, bs_cvars), (deep_pnl_cr, deep_cvars)]:
        losses = -pnl
        var_95 = np.percentile(losses, 95)
        tail = losses[losses >= var_95]
        cvar_list.append(float(np.mean(tail)) if len(tail) > 0 else float(var_95))

print("Cost Rate (bps) | BS CVaR  | Deep CVaR")
print("-" * 42)
for cr, bs_c, deep_c in zip(cost_rates, bs_cvars, deep_cvars):
    print(f"  {cr*10000:6.1f}        | {bs_c:7.4f}  | {deep_c:7.4f}")

fig, ax = plot_transaction_cost_impact(cost_rates, bs_cvars, deep_cvars)
plt.show()

In [ ]:
# Summary using the model class
model = DeepHedgingModel(config)
results = model.train_hedger()
comparison = model.compare_with_bs()
print("\n--- Method Comparison ---")
print(comparison.to_string(index=False))

print("\n--- Position Analysis (sample) ---")
positions_df = model.analyze_positions()
print(positions_df.head(15).to_string(index=False))